# Layer 1 (Maltrail) test vs the traffic of 2 UWF-ZeekData22 attacker hosts

**Part of:** the research project "Scalability of Orchestration in a Security Operation Center" (PFR 2026) —
a complement to `layer1_experiment.ipynb` (the earlier 5-pcap pilot experiment), NOT a replacement.

**Question answered by this notebook**: the paper's architecture (Figure 1) claims that "traffic that passes
Maltrail is forwarded to Layer 2". This claim had **never been tested** for the 2 new UWF-ZeekData22 hosts
(`143.88.5.12` — DNS enumeration, `143.88.7.10` — 2 different scan/probe episodes), because Layer 1 was
evaluated on the 6-pcap pilot corpus, a different population. This notebook closes that gap independently
and traceably (not just as a claim in a chat or document).

**Methodology (2 parts)**:
1. **Static blacklist** — check the 5 related IPs against the pinned Maltrail trail list
   (`trails_snapshots/`), exactly the trail list used by the other Layer 1 experiments.
2. **Maltrail behavioural heuristics** — the Maltrail sensor (`old/sensor.py`) used in this project has 6
   named heuristics besides the blacklist (`port_scanning`, `udp_scanning`, `infection`, `web_scanning`,
   `dns_exhaustion`, `long_domain`). Their thresholds and enabled/disabled status are READ DIRECTLY from
   `core/settings.py` and this project's `maltrail.conf`/`maltrail.local.conf` (not assumed), then applied to
   the raw UWF data with exactly the same windowing logic as the sensor (30-second buckets, per
   source-destination pair).

**An honest note on the method**: the available UWF-ZeekData22 data is a CSV export of Zeek conn logs, not raw
pcaps — so the Maltrail sensor (`old/sensor.py`) cannot be run directly on this traffic (unlike the 5-pcap
pilot in `layer1_experiment.ipynb`). What is done here is to apply Maltrail's threshold/window logic
*manually* to the CSV data — read directly from Maltrail's source code & config, not approximated.

**Results reference**: the project's internal Layer 2 framework document §4.5, and the caveat in the paper's
Discussion section.

## Section A — Load the raw data

Loaded directly from the raw UWF-ZeekData22 CSV shards in `../dataset/` (not from intermediate scratch
files), so this notebook can be re-run from scratch at any time.

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import csv

BASE_DIR = Path.cwd()  # run this notebook from inside the layer1-maltrail-zeek/ folder
DATASET_DIR = BASE_DIR.parent / "dataset"
MALTRAIL_DIR = BASE_DIR / "tools" / "maltrail"
TRAILS_SNAPSHOT = BASE_DIR / "trails_snapshots" / "trails_2026-08-28T0152Z_frozen.csv"

# UWF-ZeekData22 CSV columns, order confirmed directly from the file header (not third-party documentation)
UWF_COLS = ["resp_pkts","service","orig_ip_bytes","local_resp","missed_bytes","protocol",
            "duration","conn_state","dest_ip","orig_pkts","community_id","resp_ip_bytes",
            "dest_port","orig_bytes","local_orig","datetime","history","resp_bytes","uid",
            "src_port","ts","src_ip","mitre_attack_tactics"]
IDX = {c: i for i, c in enumerate(UWF_COLS)}

# The shard -> host mapping was confirmed by grepping all 6 shards directly (Layer 2 framework document §1/§4)
SHARDS_5_12 = ["part-00000-0af89d10-df53-44fd-b124-a8a496fd5023-c000.csv",
               "part-00000-5b4f5c3f-e8a9-4020-8fa1-e8985f7c27f3-c000.csv",
               "part-00000-95e0a460-e7c5-4b35-8367-c2e6fbbcf9e1-c000.csv"]
SHARD_710_EP1 = "part-00000-318611a1-7cdc-4dd0-9348-c6368917fd0c-c000.csv"   # 10 Feb, Reconnaissance
SHARD_710_EP2 = "part-00000-15e3dd03-ea76-429e-a52a-ce96a90517f9-c000.csv"   # 11 Feb, Discovery

def load_host_rows(paths, ip):
    rows = []
    for p in paths:
        with open(p, newline="", errors="replace") as f:
            reader = csv.reader(f)
            header = next(reader)
            assert header == UWF_COLS, f"unexpected header in {p}"
            for parts in reader:
                if len(parts) != len(UWF_COLS):
                    continue
                if parts[IDX["src_ip"]] != ip:
                    continue
                try:
                    ts = float(parts[IDX["ts"]])
                except ValueError:
                    continue
                rows.append(dict(ts=ts, dip=parts[IDX["dest_ip"]], dport=parts[IDX["dest_port"]],
                                  proto=parts[IDX["protocol"]], conn_state=parts[IDX["conn_state"]],
                                  service=parts[IDX["service"]], tactic=parts[IDX["mitre_attack_tactics"]],
                                  datetime=parts[IDX["datetime"]]))
    rows.sort(key=lambda r: r["ts"])
    return rows

rows_512 = load_host_rows([DATASET_DIR / s for s in SHARDS_5_12], "143.88.5.12")
rows_710_ep1 = load_host_rows([DATASET_DIR / SHARD_710_EP1], "143.88.7.10")
rows_710_ep2 = load_host_rows([DATASET_DIR / SHARD_710_EP2], "143.88.7.10")

print(f"143.88.5.12: {len(rows_512)} rows (expected 502528, framework document §1)")
print(f"143.88.7.10 Episode 1 (Reconnaissance, 10 Feb): {len(rows_710_ep1)} rows (expected 2048)")
print(f"143.88.7.10 Episode 2 (Discovery, 11 Feb): {len(rows_710_ep2)} rows (expected 2085)")

143.88.5.12: 502528 rows (expected 502528, framework document §1)
143.88.7.10 Episode 1 (Reconnaissance, 10 Feb): 2048 rows (expected 2048)
143.88.7.10 Episode 2 (Discovery, 11 Feb): 2085 rows (expected 2085)


## Section B — Static blacklist (pinned trail list)

Check the 5 related IPs against the same Maltrail trail list used by the other Layer 1 experiments
(`pin_trails_from_snapshot()` in `layer1_experiment.ipynb`).

In [2]:
IPS_TO_CHECK = ["143.88.5.12", "143.88.5.1", "143.88.7.10", "143.88.2.10", "143.88.2.12"]

matches = defaultdict(list)
n_rows = 0
with open(TRAILS_SNAPSHOT, newline="", errors="replace") as f:
    reader = csv.reader(f)
    for row in reader:
        n_rows += 1
        if not row:
            continue
        indicator = row[0]
        for ip in IPS_TO_CHECK:
            if indicator == ip:
                matches[ip].append(row)

print(f"Trail list checked: {n_rows:,} rows ({TRAILS_SNAPSHOT.name})")
for ip in IPS_TO_CHECK:
    hit = matches.get(ip, [])
    print(f"  {ip}: {'MATCH -> ' + str(hit) if hit else 'no match'}")

Trail list checked: 1,791,806 rows (trails_2026-08-28T0152Z_frozen.csv)
  143.88.5.12: no match
  143.88.5.1: no match
  143.88.7.10: no match
  143.88.2.10: no match
  143.88.2.12: no match


## Section C — Maltrail heuristic thresholds & status (read directly from the source, not assumed)

In [3]:
import re

settings_text = (MALTRAIL_DIR / "core" / "settings.py").read_text()

def read_const(name):
    m = re.search(rf"^{name}\s*=\s*(\d+)", settings_text, re.M)
    assert m, f"{name} not found in core/settings.py"
    return int(m.group(1))

PORT_SCANNING_THRESHOLD = read_const("PORT_SCANNING_THRESHOLD")
INFECTION_SCANNING_THRESHOLD = read_const("INFECTION_SCANNING_THRESHOLD")
DNS_EXHAUSTION_THRESHOLD = read_const("DNS_EXHAUSTION_THRESHOLD")
SCAN_WINDOW_SEC = 30  # sensor.py default when the config does not override it (confirmed from old/sensor.py, line ~247)

print(f"PORT_SCANNING_THRESHOLD      = {PORT_SCANNING_THRESHOLD}  (distinct ports to 1 dst_ip per window)")
print(f"INFECTION_SCANNING_THRESHOLD = {INFECTION_SCANNING_THRESHOLD}  (distinct dst_ips to 1 dst_port per window)")
print(f"DNS_EXHAUSTION_THRESHOLD     = {DNS_EXHAUSTION_THRESHOLD}  (distinct subdomains per domain per window)")
print(f"SCAN_WINDOW (default)        = {SCAN_WINDOW_SEC} seconds")
print()

for conf_name in ["maltrail.conf", "maltrail.local.conf"]:
    conf_path = MALTRAIL_DIR / conf_name
    if not conf_path.exists():
        print(f"{conf_name}: not present")
        continue
    conf_text = conf_path.read_text()
    use_h = re.search(r"^USE_HEURISTICS\s+(\S+)", conf_text, re.M)
    dis_h = re.search(r"^DISABLED_HEURISTICS\b", conf_text, re.M)  # active (not commented out) if it matches
    print(f"{conf_name}: USE_HEURISTICS={use_h.group(1) if use_h else '(not set, default)'}  "
          f"DISABLED_HEURISTICS={'SET (some heuristics disabled)' if dis_h else 'NOT set -> all heuristics enabled (default)'}")

PORT_SCANNING_THRESHOLD      = 10  (distinct ports to 1 dst_ip per window)
INFECTION_SCANNING_THRESHOLD = 32  (distinct dst_ips to 1 dst_port per window)
DNS_EXHAUSTION_THRESHOLD     = 1000  (distinct subdomains per domain per window)
SCAN_WINDOW (default)        = 30 seconds

maltrail.conf: USE_HEURISTICS=true  DISABLED_HEURISTICS=NOT set -> all heuristics enabled (default)
maltrail.local.conf: USE_HEURISTICS=true  DISABLED_HEURISTICS=NOT set -> all heuristics enabled (default)


## Section D — Apply the `port_scanning`/`udp_scanning`/`infection` logic to the raw data

Bucketing per 30-second window exactly as the sensor works (`_scan_window`, `_connect_src_dst`, `_udp_scan` in
`old/sensor.py`): key `(src_ip, dst_ip)` -> set of distinct destination ports (TCP), compared with
`PORT_SCANNING_THRESHOLD` in each window.

In [4]:
def port_scanning_check(rows, label):
    '''Apply Maltrail's port_scanning logic: per 30 s window, per (src, dst_ip), count the set of distinct ports.'''
    if not rows:
        print(f"{label}: no data")
        return
    t0 = rows[0]["ts"]
    proto_counts = Counter(r["proto"] for r in rows)
    buckets = defaultdict(lambda: defaultdict(set))
    for r in rows:
        w = int((r["ts"] - t0) // SCAN_WINDOW_SEC)
        buckets[w][r["dip"]].add(r["dport"])

    max_ports = 0
    alerts = []
    for w in sorted(buckets):
        for dip, ports in buckets[w].items():
            max_ports = max(max_ports, len(ports))
            if len(ports) > PORT_SCANNING_THRESHOLD:
                alerts.append((w, dip, len(ports)))

    distinct_dip = len(set(r["dip"] for r in rows))
    print(f"=== {label} ===")
    print(f"  {len(rows)} rows, {len(buckets)} 30-second windows, protocols: {dict(proto_counts)}")
    print(f"  distinct dst_ip: {distinct_dip}")
    print(f"  max distinct ports to 1 dst_ip within 1 window: {max_ports}  (port_scanning threshold={PORT_SCANNING_THRESHOLD})")
    print(f"  number of (window, dst_ip) pairs above the threshold: {len(alerts)}")
    if alerts:
        print(f"  examples: {alerts[:5]}")
    verdict = "WOULD BE FLAGGED by port_scanning" if alerts else "NOT flagged by port_scanning"
    print(f"  >> VERDICT: {verdict}")
    print()
    return dict(label=label, n_rows=len(rows), n_windows=len(buckets), max_ports_per_window=max_ports,
                distinct_dst_ip=distinct_dip, n_alerting_windows=len(alerts), flagged=bool(alerts))

result_ep1 = port_scanning_check(rows_710_ep1, "143.88.7.10 Episode 1 (Reconnaissance, 10 Feb)")
result_ep2 = port_scanning_check(rows_710_ep2, "143.88.7.10 Episode 2 (Discovery, 11 Feb)")
result_512 = port_scanning_check(rows_512, "143.88.5.12 (DNS enumeration)")

=== 143.88.7.10 Episode 1 (Reconnaissance, 10 Feb) ===
  2048 rows, 3 30-second windows, protocols: {'tcp': 2048}
  distinct dst_ip: 1
  max distinct ports to 1 dst_ip within 1 window: 2  (port_scanning threshold=10)
  number of (window, dst_ip) pairs above the threshold: 0
  >> VERDICT: NOT flagged by port_scanning

=== 143.88.7.10 Episode 2 (Discovery, 11 Feb) ===
  2085 rows, 3 30-second windows, protocols: {'icmp': 2, 'tcp': 2082, 'udp': 1}
  distinct dst_ip: 1
  max distinct ports to 1 dst_ip within 1 window: 1003  (port_scanning threshold=10)
  number of (window, dst_ip) pairs above the threshold: 2
  examples: [(0, '143.88.2.12', 1003), (5, '143.88.2.12', 1000)]
  >> VERDICT: WOULD BE FLAGGED by port_scanning



=== 143.88.5.12 (DNS enumeration) ===
  502528 rows, 207 30-second windows, protocols: {'udp': 495104, 'icmp': 7424}
  distinct dst_ip: 1
  max distinct ports to 1 dst_ip within 1 window: 3  (port_scanning threshold=10)
  number of (window, dst_ip) pairs above the threshold: 0
  >> VERDICT: NOT flagged by port_scanning



### D.1 — Raw rate/volume check of `143.88.5.12` (additional context, not an official Maltrail heuristic)

Maltrail has no raw-rate heuristic independent of port/IP/subdomain diversity — checked here only to show the
scale of the traffic, not to claim detection.

In [5]:
t0 = rows_512[0]["ts"]
conn_buckets = defaultdict(int)
for r in rows_512:
    w = int((r["ts"] - t0) // SCAN_WINDOW_SEC)
    conn_buckets[w] += 1

max_rate_window = max(conn_buckets.values())
avg_rate = len(rows_512) / (rows_512[-1]["ts"] - rows_512[0]["ts"])
print(f"Total queries: {len(rows_512)} in {(rows_512[-1]['ts'] - rows_512[0]['ts'])/60:.1f} minutes")
print(f"Overall average: {avg_rate:.1f} queries/second")
print(f"30-second window with the most queries: {max_rate_window} ({max_rate_window/30:.1f} queries/second)")
print()
print("None of Maltrail's 6 heuristics (port_scanning, udp_scanning, infection, "
      "web_scanning, dns_exhaustion, long_domain) measures raw rate/volume independently "
      "of port/IP/subdomain diversity -- so this pattern, although extreme in volume, PASSES "
      "all existing heuristics.")

Total queries: 502528 in 103.1 minutes
Overall average: 81.2 queries/second
30-second window with the most queries: 18176 (605.9 queries/second)

None of Maltrail's 6 heuristics (port_scanning, udp_scanning, infection, web_scanning, dns_exhaustion, long_domain) measures raw rate/volume independently of port/IP/subdomain diversity -- so this pattern, although extreme in volume, PASSES all existing heuristics.


## Section E — Limitation: `dns_exhaustion` cannot be tested with this data

This heuristic measures the diversity of **subdomain names** per domain within one window (threshold
`DNS_EXHAUSTION_THRESHOLD`), not the query rate. That field is absent from the UWF-ZeekData22 CSV export.

In [6]:
dns_relevant_fields = {c for c in UWF_COLS if "query" in c.lower() or "domain" in c.lower() or "dns" in c.lower()}
print(f"UWF-ZeekData22 columns related to DNS query/domain names: {dns_relevant_fields or '(NONE)'}")
print(f"DNS-related columns that DO exist: service={rows_512[0]['service']!r} (only a protocol label, not a domain name)")
print()
print("Conclusion: dns_exhaustion CANNOT be tested with the data we have -- \"this needs verification\" "
      "with another source (a full Zeek dns.log or a PCAP containing the real query names).")

UWF-ZeekData22 columns related to DNS query/domain names: (NONE)
DNS-related columns that DO exist: service='dns' (only a protocol label, not a domain name)

Conclusion: dns_exhaustion CANNOT be tested with the data we have -- "this needs verification" with another source (a full Zeek dns.log or a PCAP containing the real query names).


## Section F — Summary & saving the results

Saved as a permanent artefact (`results/maltrail_vs_uwf_heuristics.json`), following the same pattern as
`layer2-behavior-features/bc_results.json`, so it can be re-checked without opening this notebook.

In [7]:
import json
from datetime import datetime, timezone

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "methodology": "Maltrail threshold & window logic (PORT_SCANNING_THRESHOLD, SCAN_WINDOW) read "
                   "directly from this project's core/settings.py & maltrail.conf, applied to the raw "
                   "UWF-ZeekData22 data (CSV conn logs, not pcaps -- sensor.py is not run "
                   "directly). See the Layer 2 framework document, Section 4.5.",
    "blacklist_check": {ip: bool(matches.get(ip)) for ip in IPS_TO_CHECK},
    "thresholds": {
        "PORT_SCANNING_THRESHOLD": PORT_SCANNING_THRESHOLD,
        "INFECTION_SCANNING_THRESHOLD": INFECTION_SCANNING_THRESHOLD,
        "DNS_EXHAUSTION_THRESHOLD": DNS_EXHAUSTION_THRESHOLD,
        "SCAN_WINDOW_SEC": SCAN_WINDOW_SEC,
    },
    "heuristic_results": {
        "uwf_143.88.7.10_episode1_reconnaissance": result_ep1,
        "uwf_143.88.7.10_episode2_discovery": result_ep2,
        "uwf_143.88.5.12_dns": result_512,
    },
    "dns_exhaustion": "cannot_be_tested -- query/domain name field absent from UWF-ZeekData22 export",
    "max_dns_query_rate_per_30s_window": max_rate_window,
}

out_path = BASE_DIR / "results" / "maltrail_vs_uwf_heuristics.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(summary, indent=2))
print(f"Saved: {out_path}")
print()
print(json.dumps(summary, indent=2))

Saved: /Users/ariefikhwanwicaksono/Disertasi/PFR/beyond-the-blacklist-soc/layer1-maltrail-zeek/results/maltrail_vs_uwf_heuristics.json

{
  "generated_at_utc": "2026-11-03T05:07:53.180723+00:00",
  "methodology": "Maltrail threshold & window logic (PORT_SCANNING_THRESHOLD, SCAN_WINDOW) read directly from this project's core/settings.py & maltrail.conf, applied to the raw UWF-ZeekData22 data (CSV conn logs, not pcaps -- sensor.py is not run directly). See the Layer 2 framework document, Section 4.5.",
  "blacklist_check": {
    "143.88.5.12": false,
    "143.88.5.1": false,
    "143.88.7.10": false,
    "143.88.2.10": false,
    "143.88.2.12": false
  },
  "thresholds": {
    "PORT_SCANNING_THRESHOLD": 10,
    "INFECTION_SCANNING_THRESHOLD": 32,
    "DNS_EXHAUSTION_THRESHOLD": 1000,
    "SCAN_WINDOW_SEC": 30
  },
  "heuristic_results": {
    "uwf_143.88.7.10_episode1_reconnaissance": {
      "label": "143.88.7.10 Episode 1 (Reconnaissance, 10 Feb)",
      "n_rows": 2048,
      "n_window